In [1]:
from Util.Problems import Problem, solution

import Util.math_functions as mathf

import numpy as np
from itertools import combinations

class P030(Problem):
    number = 30
    title = "Digit Fifth Powers"
    description = """<p>Surprisingly there are only three numbers that can be written as the sum of fourth powers of their digits:
$$\\begin{align}
1634 &= 1^4 + 6^4 + 3^4 + 4^4\\\\
8208 &= 8^4 + 2^4 + 0^4 + 8^4\\\\
9474 &= 9^4 + 4^4 + 7^4 + 4^4
\\end{align}$$
</p><p class="smaller">As $1 = 1^4$ is not a sum it is not included.</p><p>The sum of these numbers is $1634 + 8208 + 9474 = 19316$.</p><p>Find the sum of all the numbers that can be written as the sum of fifth powers of their digits.</p>"""

In [2]:
p = P030()
p.describe()

## Problem 30: Digit Fifth Powers

<p>Surprisingly there are only three numbers that can be written as the sum of fourth powers of their digits:
$$\begin{align}
1634 &= 1^4 + 6^4 + 3^4 + 4^4\\
8208 &= 8^4 + 2^4 + 0^4 + 8^4\\
9474 &= 9^4 + 4^4 + 7^4 + 4^4
\end{align}$$
</p><p class="smaller">As $1 = 1^4$ is not a sum it is not included.</p><p>The sum of these numbers is $1634 + 8208 + 9474 = 19316$.</p><p>Find the sum of all the numbers that can be written as the sum of fifth powers of their digits.</p>

### Solution notes

As $9^5=59.049$, we know no number of $7$ digits can be written as the sum of the fifth power of its digits. This is because $59.049 \times 7 = 413.343$ which is only $6$ digits. Therefore, we will only need to check for numbers between 2 and 6 digits in length (as 1 digit is not a sum). If we include 0 in the digits, we can simply check all combinations of 6 digits to find which results meet our criteria. To make this faster, we will pre-compute the powers of 5 once, and simply look up the value in our array in our loops.

This is some of the ugliest brutest force code I have ever written, so a better implementation will follow, but for now the sextuple nested for loop does do the trick. There are many ways of improving though, which I will get to in due time.

In [3]:
@solution(P030, max_tests= 1, first=True, make_fast=True)
def brute_force():
    lookup_table = np.zeros(10, dtype=np.int_)
    for d in range(10):
        lookup_table[d] = d ** 5

    results = []

    for a, a_5 in enumerate(lookup_table):
        for b, b_5 in enumerate(lookup_table):
            for c, c_5 in enumerate(lookup_table):
                for d, d_5 in enumerate(lookup_table):
                    for e, e_5 in enumerate(lookup_table):
                        for f, f_5 in enumerate(lookup_table):
                            total_sum = a_5 + b_5 + c_5 + d_5 + e_5 + f_5
                            if total_sum < 10 or total_sum in results:
                                continue
                            possible_digits = np.zeros(10, dtype=np.int8)
                            for num in [a, b, c, d, e, f]:
                                possible_digits[num] += 1
                            for digit, occurrence in enumerate(possible_digits):
                                if digit == 0:
                                    continue
                                if occurrence != str(total_sum).count(str(digit)):
                                    break
                            else:
                                results.append(total_sum)

    return sum(results)

In [4]:
p.test_once("brute_force")

443839 found after a separate test in 385.266500 ms by brute_force (first)


We can prevent a lot of double work in this brute force method by having each inner loop start at the digit of the previous loop. This will result in a sorted list of digits, so we could check for example $001234$, but we will not check $012340$. For each combination of digits, we are only checking if the sum of the fifth powers can be written by the same digits in the combination. If so, we save the sum. The order the digits were in is irrelevant. This still feels clunky, but the inner loop is only run 5005 times, which is the amount of combinations we have. $\frac{(n+k-1)!}{(n-1)!k!} = \frac{(10+6-1)!}{(10-1)!6!}=\frac{15!}{9!6!}=5005$

In [5]:
@solution(P030, max_tests= 100, make_fast=True)
def optimised_brute_force():
    lookup_table = np.zeros((10, 2), dtype=np.int_)
    for d in range(10):
        lookup_table[d][0] = d
        lookup_table[d][1] = d ** 5

    results = []

    iterations = 0

    for a, a_5 in lookup_table:
        for b, b_5 in lookup_table[a:]:
            for c, c_5 in lookup_table[b:]:
                for d, d_5 in lookup_table[c:]:
                    for e, e_5 in lookup_table[d:]:
                        for f, f_5 in lookup_table[e:]:
                            iterations += 1
                            total_sum = a_5 + b_5 + c_5 + d_5 + e_5 + f_5
                            if total_sum < 10 or total_sum in results:
                                continue
                            possible_digits = np.zeros(10, dtype=np.int8)
                            for num in [a, b, c, d, e, f]:
                                possible_digits[num] += 1
                            for digit, occurrence in enumerate(possible_digits):
                                if digit == 0:
                                    continue
                                if occurrence != str(total_sum).count(str(digit)):
                                    break
                            else:
                                results.append(total_sum)
    return sum(results)

In [6]:
p.test_all()

443839 found after 1 test in 390.202600 ms by brute_force (first)
443839 found after 100 tests in 1.972366 ms by optimised_brute_force
